# 3. Governance Structure Setup (Dr. Egeria)

This workbook runs the Dr. Egeria markdown plan to setup organizational, semantic, and blueprint definitions, and creates the forecasting governance community.

In [ ]:
import sys
import asyncio
sys.path.insert(0, '/Users/dwolfson/localGit/egeria-v6/egeria-advisor/data/repos/egeria-python')

from pyegeria import EgeriaTech, settings, config_logging
from md_processing.dr_egeria import process_md_file_v2

config_logging()
app_config = settings.Environment

EGERIA_USER = 'erinoverview'
EGERIA_USER_PASSWORD = 'secret'

client = EgeriaTech(app_config.egeria_view_server,
                    app_config.egeria_view_server_url,
                    EGERIA_USER, EGERIA_USER_PASSWORD)
token = client.create_egeria_bearer_token(client.user_id, client.user_pwd)

print("Connected and authenticated.")

## 1. Run Dr. Egeria Governance Plan

We execute the `sales_forecast_governance_plan.md` plan to register projects, people, teams, glossary terms, blueprints, and supply chains in Egeria.

In [ ]:
try:
    # We run the processing v2 parser asynchronously
    # directive='process' commits changes; directive='validate' runs dry-run
    await process_md_file_v2(
        input_file='sales_forecast_governance_plan.md',
        output_folder='dr-egeria-outbox',
        directive='process',
        client=client,
        parse_summary='all',
        attribute_logs='info',
        usage_level='Basic'
    )
    print("Dr. Egeria plan executed successfully.")
except Exception as e:
    print("Error executing Dr. Egeria plan:", e)

## 2. Create the Sales Forecasting Governance Community

Since Community creation is not natively supported by Dr. Egeria, we call the `CommunityMatters` OMVS client directly to build the community.

In [ ]:
from pyegeria.omvs.community_matters_omvs import CommunityMatters

comm_client = CommunityMatters(app_config.egeria_view_server,
                               app_config.egeria_view_server_url,
                               EGERIA_USER, EGERIA_USER_PASSWORD, token)

try:
    # Define community properties
    comm_body = {
        "class": "NewElementRequestBody",
        "properties": {
            "class": "CommunityProperties",
            "qualifiedName": "Community::SalesForecastingCommunity",
            "displayName": "Sales Forecasting Governance Community",
            "description": "Community to discuss, define, and govern sales forecasting data, schemas, and models across regions.",
            "category": "Governance Community"
        }
    }
    comm_guid = comm_client.create_community(body=comm_body)
    print(f"Sales Forecasting Community created. GUID: {comm_guid}")
except Exception as e:
    print("Error creating community:", e)